# Construyendo el Almacén de Datos de la Liga

**El Problema:**
Ustedes son los Arquitectos de Datos de una agencia internacional de *scouting*. Cada fin de semana, el sistema transaccional (OLTP) de los estadios genera un archivo plano masivo (`players_data-2025_2026.csv`) con todos los eventos registrados. 

Si el director de la agencia intenta calcular métricas históricas conectándose directamente a este archivo crudo, el sistema colapsa por redundancia y falta de indexación analítica.

**Su Misión:**
Construir una tubería **ETL (Extract, Transform, Load)** que tome este archivo transaccional y lo convierta en un **Esquema de Estrella (OLAP)** en memoria. Al final, deberán ejecutar un cubo analítico (*Roll-up*) para responder una pregunta de negocio estratégica.

In [1]:
import pandas as pd
import numpy as np

print("--- FASE 1: EXTRACT (Extracción Transaccional) ---")
# Simulamos el robot nocturno que extrae los datos de producción.
df_raw = pd.read_csv('./datasets/players_data-2025_2026.csv')

print("Datos crudos extraídos del sistema OLTP:")
display(df_raw.head(3))

--- FASE 1: EXTRACT (Extracción Transaccional) ---
Datos crudos extraídos del sistema OLTP:


,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,CrdY_stats_misc,CrdR_stats_misc,2CrdY,Fls,Fld,Off,Crs,Int,TklW,OG
0,1,Brenden Aaronson,us USA,"MF,FW",Leeds United,eng Premier League,24.0,2000.0,37,30,...,3,0,0,20,51,5,37,17,27,0
1,2,Jerome Abbey,eng ENG,MF,Wolves,eng Premier League,15.0,2009.0,1,0,...,0,0,0,0,0,1,0,1,0,0
2,3,Zach Abbott,eng ENG,DF,Nottingham Forest,eng Premier League,19.0,2006.0,3,2,...,0,0,0,3,1,0,3,2,4,0


---
### FASE 2: TRANSFORM (Modelado del Esquema de Estrella)

El archivo plano tiene demasiado texto mezclado con métricas. Vamos a separarlo arquitectónicamente. Primero, crearemos nuestras **Tablas de Dimensión** (El Contexto).

**Reto 1:** Extraigan los datos únicos para crear una dimensión de Equipos y una dimensión de Jugadores. Generen IDs numéricos artificiales para optimizar la base analítica.

In [2]:
print("--- FASE 2A: TRANSFORM (Tablas de Dimensión) ---")

rows = len(df_raw.index)

dim_equipo = df_raw[['Squad', 'Comp']].drop_duplicates().reset_index(drop=True)
dim_equipo['id_equipo'] = pd.Series(range(1, rows))
dim_jugador = df_raw[['Player', 'Pos', 'Age']].drop_duplicates().reset_index(drop=True)
dim_jugador['id_jugador'] = pd.Series(range(1, rows))

print("Dimensión Equipo (Muestra):")
display(dim_equipo.head(3))
print("\nDimensión Jugador (Muestra):")
display(dim_jugador.head(3))

--- FASE 2A: TRANSFORM (Tablas de Dimensión) ---
Dimensión Equipo (Muestra):


,Squad,Comp,id_equipo
0,Leeds United,eng Premier League,1
1,Wolves,eng Premier League,2
2,Nottingham Forest,eng Premier League,3



Dimensión Jugador (Muestra):


,Player,Pos,Age,id_jugador
0,Brenden Aaronson,"MF,FW",24.0,1
1,Jerome Abbey,MF,15.0,2
2,Zach Abbott,DF,19.0,3


Ahora construiremos el corazón del Data Warehouse: **La Tabla de Hechos (`Fact_Rendimiento`)**.
Recuerden la regla de oro: La tabla de hechos NO debe contener texto. Solo debe tener los IDs de las dimensiones que acabamos de crear y las métricas numéricas puras (goles, asistencias, minutos).

In [3]:
print("--- FASE 2B: TRANSFORM (Tabla de Hechos) ---")

df_temp = df_raw.merge(dim_equipo, on=["Squad"], how='left')
df_temp = df_temp.merge(dim_jugador, on=['Player'], how='left')
fact_rendimiento = df_temp[['id_equipo', 'id_jugador', 'Gls', 'Ast', 'Min']]

print("Tabla de Hechos (Métrica Pura):")
display(fact_rendimiento.head(3))

--- FASE 2B: TRANSFORM (Tabla de Hechos) ---
Tabla de Hechos (Métrica Pura):


,id_equipo,id_jugador,Gls,Ast,Min
0,1,1,4,5,2449
1,2,2,0,0,17
2,3,3,0,0,164


---
### FASE 3: LOAD Y CONSULTA OLAP (Roll-up)

Nuestra arquitectura está lista. Hemos separado el almacenamiento para que sea eficiente. 

De pronto, el Director Técnico hace una pregunta que requiere un análisis multidimensional masivo (*Roll-up*):
*"¿Cuál es la Liga de fútbol que tiene la mayor cantidad de Goles Totales acumulados, y cuántos minutos en promedio juegan los jugadores de esa liga?"*

**Reto Final:** 
Para responder, no pueden usar el archivo `df_raw`. Deben simular el comportamiento de un motor OLAP: unan la Tabla de Hechos con la Dimensión Equipos y ejecuten una agrupación.

In [16]:
print("--- FASE 3: CUBO OLAP (Roll-up Analítico) ---")

cubo_olap = fact_rendimiento.merge(dim_equipo, on=["id_equipo"], how='left')
analisis_ligas = cubo_olap.groupby(["Squad"]).agg({"Gls": 'sum', "Min": "mean"}).sort_values(by='Gls', ascending=False)

print("Respuesta Analítica para el Director Técnico:")
display(analisis_ligas)

--- FASE 3: CUBO OLAP (Roll-up Analítico) ---
Respuesta Analítica para el Director Técnico:


,Gls,Min
Squad,,
Bayern Munich,119,1050.125000
Barcelona,94,1297.172414
Inter,85,1297.241379
Manchester City,83,1374.935484
Roma,79,1210.342857
...,...,...
Lecce,28,1240.129032
Hellas Verona,27,1091.250000
Oviedo,26,1207.387097


---
### 🧠 Defensa del Diseño Arquitectónico (Responde aquí)

Como ingenieros de datos, acaban de transformar un archivo plano en un modelo de estrella y luego volvieron a unir las tablas para hacer la consulta OLAP. 

1. Si al final tuvimos que hacer un `merge()` (JOIN) para responder la pregunta del Director, ¿por qué nos tomamos la molestia de separar los datos en Hechos y Dimensiones en lugar de dejar el `df_raw` gigante en el servidor analítico desde el principio? 
2. ¿Qué ventaja física (de disco o memoria) tiene aislar las columnas de texto (`player_name`, `team_name`) en dimensiones separadas?

> **Tu Respuesta:** El separar los datos en Hechos y Dimensiones hace más fácil la busqueda de información. Si usaramos df_raw estaríamos buscando también con más de 100 columnas que realmente no necesitabamos, con la separación se reduce la cantidad de columnas y se vuelve más legible el resultado. Además, tenemos una ventaja al aislar el jugador del equipo, podemos solo tener su id en caso de no ser necesarios en el análisis y realizar una busqueda rápdia en caso de que si. Esto también ayuda a que un solo dataframe no se vuelva muy grande.  